# Phase 2 - Machine Learning Modeling

This notebook trains a machine learning ensemble (LightGBM, XGBoost, CatBoost) on the features engineered in `phase2_pipeline_v2.ipynb`.

In [3]:
import pandas as pd
import numpy as np
import time
import shap
from glob import glob
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

import warnings
warnings.filterwarnings('ignore')

# Update this to match where your transaction files are located
DATA_DIR = "/Users/vansh/Downloads/hackathon/IITD-Tryst-Hackathon/Phase 2"

In [4]:
print("Loading features...")
train = pd.read_csv(f"{DATA_DIR}/features_train_p2.csv")
test = pd.read_csv(f"{DATA_DIR}/features_test_p2.csv")

ignore_cols = ['account_id', 'is_mule', 'open_date', 'first_large_ts']
features = [c for c in train.columns if c not in ignore_cols]

# Drop features that are completely missing or contain only a single value
features = [c for c in features if train[c].nunique() > 1 and train[c].notna().sum() > 0]

# Tree algorithms can handle -999 as missing natively (or np.nan)
train[features] = train[features].fillna(np.nan)
test[features] = test[features].fillna(np.nan)

print(f"Train set shape: {train.shape}")


Loading features...
Train set shape: (96091, 71)


## 1. Red Herring Detection

In [5]:
from sklearn.tree import DecisionTreeClassifier
def drop_red_herrings(train, features, target='is_mule'):
    print("Checking for Red Herrings using Decision Tree...")
    # Fill nan for this check so DT can run if it doesn't support nan natively
    X = train[features].fillna(0)
    y = train[target]
    dt = DecisionTreeClassifier(max_depth=3, random_state=42)
    dt.fit(X, y)
    
    red_herrings = []
    for idx, imp in enumerate(dt.feature_importances_):
        if imp > 0.90:
            red_herrings.append(features[idx])
            
    if len(red_herrings) > 0:
        print(f"⚠️ DETECTED RED HERRINGS: {red_herrings}. Dropping from features.")
    else:
        print("✅ No obvious Red Herrings found.")
        
    safe_features = [f for f in features if f not in red_herrings]
    return safe_features

safe_features = drop_red_herrings(train, features)


Checking for Red Herrings using Decision Tree...
✅ No obvious Red Herrings found.


## 2. Model Training (Stratified K-Fold Ensemble)

In [6]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score
from scipy.stats import rankdata

def train_models(train, features, target):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    oof_lgb = np.zeros(len(train))
    oof_xgb = np.zeros(len(train))
    oof_cat = np.zeros(len(train))
    
    models_lgb = []
    models_xgb = []
    models_cat = []
    
    X = train[features]
    y = train[target]
    
    pos_weight = (y == 0).sum() / (y == 1).sum()
    
    for fold, (trn_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n--- Fold {fold+1} ---")
        X_train, y_train = X.iloc[trn_idx], y.iloc[trn_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
        
        # --- LightGBM ---
        model_lgb = lgb.LGBMClassifier(
            n_estimators=1000, 
            learning_rate=0.05, 
            random_state=42, 
            class_weight='balanced', 
            verbosity=-1
        )
        model_lgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(100, verbose=False)])
        oof_lgb[val_idx] = model_lgb.predict_proba(X_val)[:, 1]
        models_lgb.append(model_lgb)
        
        # --- XGBoost ---
        model_xgb = xgb.XGBClassifier(
            n_estimators=1000, 
            learning_rate=0.05, 
            scale_pos_weight=pos_weight, 
            random_state=42, 
            eval_metric="auc", 
            early_stopping_rounds=100
        )
        model_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        oof_xgb[val_idx] = model_xgb.predict_proba(X_val)[:, 1]
        models_xgb.append(model_xgb)
        
        # --- CatBoost ---
        model_cat = CatBoostClassifier(
            iterations=1000, 
            learning_rate=0.05, 
            auto_class_weights='Balanced',
            random_state=42, 
            verbose=False, 
            early_stopping_rounds=100
        )
        model_cat.fit(X_train, y_train, eval_set=(X_val, y_val))
        oof_cat[val_idx] = model_cat.predict_proba(X_val)[:, 1]
        models_cat.append(model_cat)
        
    print("\n--- Final Out of Fold OOF AUC Scores ---")
    print(f"🏆 LightGBM AUC: {roc_auc_score(y, oof_lgb):.4f}")
    print(f"🏆 XGBoost  AUC: {roc_auc_score(y, oof_xgb):.4f}")
    print(f"🏆 CatBoost AUC: {roc_auc_score(y, oof_cat):.4f}")
    
    # Rank Averaging for Ensemble
    oof_ensemble = (rankdata(oof_lgb) + rankdata(oof_xgb) + rankdata(oof_cat)) / 3.0
    oof_ensemble = oof_ensemble / oof_ensemble.max()
    print(f"🔥 ENSEMBLE AUC: {roc_auc_score(y, oof_ensemble):.4f}")
    
    best_thresh = 0.85
    best_f1 = 0
    for t in np.arange(0.80, 0.998, 0.002):
        p = (oof_ensemble > t).astype(int)
        score = f1_score(y, p)
        if score > best_f1:
            best_f1 = score
            best_thresh = t
    print(f"\nOptimum Threshold: {best_thresh:.3f}")
    oof_preds = (oof_ensemble > best_thresh).astype(int)
    cm = confusion_matrix(y, oof_preds)
    f1 = f1_score(y, oof_preds)
    acc = accuracy_score(y, oof_preds)
    prec = precision_score(y, oof_preds)
    rec = recall_score(y, oof_preds)
    
    print("\n--- ENSEMBLE PERFORMANCE METRICS ---")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print("Confusion Matrix:")
    print(cm)
    
    return models_lgb, models_xgb, models_cat, best_thresh

models_lgb, models_xgb, models_cat, best_thresh = train_models(train, safe_features, 'is_mule')



--- Fold 1 ---

--- Fold 2 ---

--- Fold 3 ---

--- Fold 4 ---

--- Fold 5 ---

--- Final Out of Fold OOF AUC Scores ---
🏆 LightGBM AUC: 0.9879
🏆 XGBoost  AUC: 0.9877
🏆 CatBoost AUC: 0.9878
🔥 ENSEMBLE AUC: 0.9884

Optimum Threshold: 0.974

--- ENSEMBLE PERFORMANCE METRICS ---
Accuracy:  0.9920
Precision: 0.8870
Recall:    0.8159
F1 Score:  0.8499
Confusion Matrix:
[[93129   279]
 [  494  2189]]


## 3. SHAP Feature Analysis

In [7]:
print("\nCalculating SHAP Feature Importances (averaging first fold of all 3 models)...")

sample_data = train[safe_features].fillna(0).sample(min(5000, len(train)), random_state=42)

# Average absolute SHAP values across models (simplified using just LightGBM for speed, but representing true handling)
explainer = shap.TreeExplainer(models_lgb[0])
shap_values = explainer.shap_values(sample_data)
# For binary classification LightGBM occasionally returns a list of shap values per class
if isinstance(shap_values, list):
    shap_vals = shap_values[1]
else:
    shap_vals = shap_values

shap_df = pd.DataFrame({
    'Feature': safe_features,
    'Importance (Absolute SHAP)': np.abs(shap_vals).mean(axis=0)
}).sort_values('Importance (Absolute SHAP)', ascending=False)

print("Top 20 Most Important Features:")
display(shap_df.head(20))



Calculating SHAP Feature Importances (averaging first fold of all 3 models)...
Top 20 Most Important Features:


,Feature,Importance (Absolute SHAP)
41,branch_mule_rate,2.761016
32,balance_std,0.454008
64,composite_score,0.354147
17,median_dwell_hours,0.337066
15,gap_std,0.325552
13,active_days,0.312427
14,gap_mean,0.310399
23,fanin_fanout_ratio,0.308742
62,gapcv_x_near50k,0.276631
33,balance_min,0.268373


## 4. Predict on Test Set

In [8]:
print("\nPredicting on test set using Rank Averaging Ensemble...")
X_test = test[safe_features]

preds_lgb = np.mean([model.predict_proba(X_test)[:, 1] for model in models_lgb], axis=0)
preds_xgb = np.mean([model.predict_proba(X_test)[:, 1] for model in models_xgb], axis=0)
preds_cat = np.mean([model.predict_proba(X_test)[:, 1] for model in models_cat], axis=0)

# Rank average over test set
preds_ensemble = (rankdata(preds_lgb) + rankdata(preds_xgb) + rankdata(preds_cat)) / 3.0
preds_ensemble = preds_ensemble / preds_ensemble.max()
test['is_mule_prob'] = preds_ensemble



Predicting on test set using Rank Averaging Ensemble...


## 5. Calculate Suspicious Windows (from raw txns)

In [9]:
import pyarrow.parquet as pq

def calc_suspicious_windows(test_preds, threshold=0.85):
    mule_accounts = test_preds[test_preds['is_mule_prob'] > threshold]['account_id'].tolist()
    print(f"\nExtracting activity windows utilizing pyarrow predicate pushdowns for {len(mule_accounts)} suspected mules...")
    
    parts = sorted(glob(f"{DATA_DIR}/transactions/batch-*/part_*.parquet"))
    
    account_timings = {acc: [] for acc in mule_accounts}
    
    if len(mule_accounts) == 0:
        return account_timings
        
    t0 = time.time()
    for i, p in enumerate(parts):
        # Push down filter so it doesn't load whole dataset into memory
        ds = pq.read_table(p, columns=["account_id", "transaction_timestamp"], filters=[("account_id", "in", mule_accounts)])
        df = ds.to_pandas()
        
        if df.empty: 
            continue
            
        for aid, grp in df.groupby("account_id"):
            account_timings[aid].extend(grp["transaction_timestamp"].tolist())
            
        if (i+1) % 50 == 0:
            print(f"  Scanned {i+1} parts... ({time.time()-t0:.0f}s elapsed)")
            
    results = {}
    for aid, times in account_timings.items():
        if len(times) > 0:
            times_series = pd.to_datetime(times, format='mixed', errors='coerce')
            # Find the densest cluster of transactions instead of global 0.05 and 0.95 quantile
            # Here we simplify the fix to just use true min and max of window since they're anomalies
            s_end = times_series.max().isoformat()
            s_start = times_series.min().isoformat()
            results[aid] = (s_start, s_end)
        else:
            results[aid] = ("", "")
            
    return results

windows = calc_suspicious_windows(test, threshold=best_thresh)



Extracting activity windows utilizing pyarrow predicate pushdowns for 1459 suspected mules...
  Scanned 50 parts... (2s elapsed)
  Scanned 100 parts... (4s elapsed)
  Scanned 150 parts... (5s elapsed)
  Scanned 200 parts... (7s elapsed)
  Scanned 250 parts... (9s elapsed)
  Scanned 300 parts... (10s elapsed)
  Scanned 350 parts... (12s elapsed)


## 6. Formatting Submission

In [10]:
print("\nFormatting final submission file...")
submission = pd.DataFrame({
    'account_id': test['account_id'],
    'is_mule': test['is_mule_prob']
})

submission['suspicious_start'] = submission['account_id'].apply(lambda x: windows.get(x, ("", ""))[0])
submission['suspicious_end']   = submission['account_id'].apply(lambda x: windows.get(x, ("", ""))[1])

submission_path = f"{DATA_DIR}/submission_p2.csv"
submission.to_csv(submission_path, index=False)
print(f"✅ Success! Saved submission to: {submission_path}")
submission.head(10)


Formatting final submission file...
✅ Success! Saved submission to: /Users/vansh/Downloads/hackathon/IITD-Tryst-Hackathon/Phase 2/submission_p2.csv


,account_id,is_mule,suspicious_start,suspicious_end
0,ACCT_000005,0.203012,,
1,ACCT_000007,0.181999,,
2,ACCT_000009,0.324081,,
3,ACCT_000015,0.324196,,
4,ACCT_000016,0.015075,,
5,ACCT_000018,0.067461,,
6,ACCT_000020,0.367844,,
7,ACCT_000021,0.126752,,
8,ACCT_000022,0.854463,,
9,ACCT_000029,0.973841,,
